# CareCost AI – Houston Healthcare Cost & Insurance Navigator

## Project Goal
Build a simple AI-driven healthcare application that helps users compare
estimated healthcare costs and out-of-pocket expenses across selected
Houston hospitals based on procedure and insurance information.

## Project Roadmap
1. Collect 3 Houston hospitals pricing data(1. Houston Methodist Hospital, Memorial Hermann,Baylor St. Luke’s Medical Center)
2. Collect insurance plan and cost-sharing data
3. Clean and prepare the datasets
4. Select hospitals and common 5 medical services

   **Medical Services Selected for Comparison:** MRI Brain without contrast (CPT 70551) | CT Chest without contrast (CPT 71250) | Chest X-ray, 2 views (CPT 71046) | Colonoscopy with biopsy (CPT 45380) | Emergency Visit, Level 3 (CPT 99283).

   * *Total:** 5 medical services compared across 3 Houston hospitals.


5. Merge hospital pricing and insurance information
6. Build the healthcare cost calculation logic
7. Add AI/NLP to understand user questions
8. Create the user interface
9. Test the application
10. Add safety/estimate disclaimer
11. Document the code
12. Upload the final project to GitHub

In [1]:
# CareCost AI - Houston
# Import libraries needed for data analysis

import pandas as pd
import numpy as np

print("CareCost AI project started successfully!")

CareCost AI project started successfully!


In [2]:
# Connect Google Drive to CareCost AI

from google.colab import drive
import os

drive.mount("/content/drive")

data_folder = "/content/drive/MyDrive/CareCost_AI/data"

print("Google Drive connected.")
print("Hospital data folder:", data_folder)

Mounted at /content/drive
Google Drive connected.
Hospital data folder: /content/drive/MyDrive/CareCost_AI/data


### Step 1: Data Collection and Sources

This project uses publicly available hospital price-transparency datasets from three Houston hospitals:

- Houston Methodist
- Memorial Hermann–Texas Medical Center
- Baylor St. Luke's Medical Center

The original JSON and CSV files were downloaded from hospital price-transparency resources and stored in Google Drive.

**Data included:** Medical service descriptions, billing codes, gross charges, discounted cash prices, insurance companies, insurance plans, and negotiated rates.

**Reproducibility:** The original hospital datasets are not embedded in this notebook. To run the project, users must obtain the source files and place them in the required data folder.

In [ ]:
import json

# Load Houston Methodist hospital pricing data
methodist_file = "/content/drive/MyDrive/CareCost_AI/data/74110155_the-methodist-hospital_standardcharges.json"

with open(methodist_file, "r", encoding="utf-8-sig") as f:
    methodist_data = json.load(f)

print("File loaded successfully!")
print("Data type:", type(methodist_data))
print("Main sections:", methodist_data.keys())

File loaded successfully!
Data type: <class 'dict'>
Main sections: dict_keys(['hospital_name', 'last_updated_on', 'version', 'location_name', 'hospital_address', 'license_information', 'type_2_npi', 'attestation', 'modifier_information', 'standard_charge_information'])


In [ ]:
# Show the main sections of the Methodist dataset
for key in methodist_data.keys():
    print(key)

hospital_name
last_updated_on
version
location_name
hospital_address
license_information
type_2_npi
attestation
modifier_information
standard_charge_information


In [ ]:
# Inspect the hospital pricing section
charges = methodist_data["standard_charge_information"]

print("Type:", type(charges))
print("Number of records:", len(charges))

# Show the first record
print(charges[0])

Type: <class 'list'>
Number of records: 29585
{'description': 'AMOXICILLIN 250 MG-POTASSIUM CLAVULANATE 125 MG TABLET [22992]', 'drug_information': {'unit': 1.0, 'type': 'EA'}, 'code_information': [{'code': '00781187431', 'type': 'NDC'}], 'standard_charges': [{'gross_charge': 60.0, 'discounted_cash': 30.0, 'setting': 'both', 'billing_class': 'facility', 'additional_generic_notes': 'Gross Charge Type: Pharmacy | NDC EA = 1 EACH'}]}


In [ ]:
# Show the fields available in one pricing record
print(charges[0].keys())

dict_keys(['description', 'drug_information', 'code_information', 'standard_charges'])


In [ ]:
# Inspect pricing information for the first record
print(type(charges[0]["standard_charges"]))
print(charges[0]["standard_charges"])

<class 'list'>
[{'gross_charge': 60.0, 'discounted_cash': 30.0, 'setting': 'both', 'billing_class': 'facility', 'additional_generic_notes': 'Gross Charge Type: Pharmacy | NDC EA = 1 EACH'}]


In [ ]:
# Find all field names used inside standard_charges
charge_fields = set()

for record in charges:
    for charge in record.get("standard_charges", []):
        charge_fields.update(charge.keys())

print("Fields found in standard_charges:")
for field in sorted(charge_fields):
    print(field)

Fields found in standard_charges:
additional_generic_notes
billing_class
discounted_cash
gross_charge
maximum
minimum
payers_information
setting


In [ ]:
# Find the first record containing payer/insurance information
for record in charges:
    for charge in record.get("standard_charges", []):
        if charge.get("payers_information"):
            print("Service:", record.get("description"))
            print("Payer information:")
            print(charge["payers_information"])
            break
    else:
        continue
    break

Service: Incision/fixation of femur
Payer information:
[{'payer_name': 'Aetna', 'plan_name': 'Hmo/Pos/Ppo', 'methodology': 'fee schedule', 'standard_charge_dollar': 7542.0}, {'payer_name': 'Aetna', 'plan_name': 'Medicare Managed Care - Hmo/Ppo', 'methodology': 'fee schedule', 'standard_charge_dollar': 7398.62, 'additional_payer_notes': 'APC'}, {'payer_name': 'Aetna', 'plan_name': 'New Business Discount Other Commercial Plan', 'methodology': 'fee schedule', 'standard_charge_dollar': 7316.0}, {'payer_name': 'Bcbs', 'plan_name': 'All Commercial Plans', 'methodology': 'fee schedule', 'standard_charge_dollar': 3397.96}, {'payer_name': 'Bcbs', 'plan_name': 'Medicare Managed Care - Hmo', 'methodology': 'fee schedule', 'standard_charge_dollar': 7289.28, 'additional_payer_notes': 'APC'}, {'payer_name': 'Cigna', 'plan_name': 'Local Plus/Surefit Hmo/Ppo', 'methodology': 'fee schedule', 'standard_charge_dollar': 15912.0}, {'payer_name': 'Cigna', 'plan_name': 'Network C-24 Hmo/Ppo', 'methodology': 

In [ ]:
payer = charge["payers_information"][0]

for key, value in payer.items():
    print(f"{key}: {value}")

payer_name: Aetna
plan_name: Hmo/Pos/Ppo
methodology: fee schedule
standard_charge_dollar: 7542.0


In [ ]:
# Load Memorial Hermann pricing dataset
import pandas as pd
import glob
import os

data_folder = "/content/drive/MyDrive/CareCost_AI/data"

# Find the Memorial Hermann CSV file
memorial_file = glob.glob(data_folder + "/*memorial-hermann*.csv")[0]

# First 2 rows are hospital metadata; actual pricing table starts on row 3
memorial_df = pd.read_csv(
    memorial_file,
    skiprows=2,
    low_memory=False
)

print("Memorial Hermann dataset loaded successfully!")
print("Rows:", memorial_df.shape[0])
print("Columns:", memorial_df.shape[1])

print("\nColumn names:")
for column in memorial_df.columns:
    print(column)

Memorial Hermann dataset loaded successfully!
Rows: 131481
Columns: 27

Column names:
description
code|1
code|1|type
code|2
code|2|type
code|3
code|3|type
billing_class
setting
drug_unit_of_measurement
drug_type_of_measurement
modifiers
standard_charge|gross
standard_charge|discounted_cash
payer_name
plan_name
standard_charge|negotiated_dollar
standard_charge|negotiated_percentage
standard_charge|negotiated_algorithm
median_amount
10th_percentile
90th_percentile
count
standard_charge|methodology
standard_charge|min
standard_charge|max
additional_generic_notes


In [ ]:
# Load and inspect Baylor St. Luke's pricing dataset
import json
import glob
import os

data_folder = "/content/drive/MyDrive/CareCost_AI/data"

# Find Baylor St. Luke's JSON file
baylor_file = glob.glob(data_folder + "/*st-luk*.json")[0]

with open(baylor_file, "r", encoding="utf-8-sig") as f:
    baylor_data = json.load(f)

print("Baylor St. Luke's dataset loaded successfully!")
print("File:", os.path.basename(baylor_file))
print("Data type:", type(baylor_data))

print("\nMain sections:")
for key in baylor_data.keys():
    print(key)

# Inspect pricing section if available
if "standard_charge_information" in baylor_data:
    baylor_charges = baylor_data["standard_charge_information"]
    print("\nNumber of pricing records:", len(baylor_charges))
    print("\nFields in one pricing record:")
    print(baylor_charges[0].keys())

Baylor St. Luke's dataset loaded successfully!
File: 741161938-1184622847_chi-st-lukes-health-baylor-college-of-medicine-medical-center_standardcharges.json
Data type: <class 'dict'>

Main sections:
hospital_name
last_updated_on
version
location_name
hospital_address
license_information
type_2_npi
attestation
standard_charge_information
modifier_information
general_contract_provisions

Number of pricing records: 45609

Fields in one pricing record:
dict_keys(['description', 'code_information', 'standard_charges'])


### Step 2: Data Cleaning and Integration

**Objective:** Prepare hospital datasets with different formats for comparison.

**Process:**
1. Read the JSON and CSV hospital files.
2. Extract service descriptions, billing codes, cash prices, insurance companies, plans, and negotiated rates.
3. Standardize the column names across all three hospitals.
4. Combine the prepared records into one Pandas DataFrame.
5. Convert price fields into numerical values and identify available pricing records.

**Result:** A combined dataset containing 2,229,434 records from three hospitals, with 1,997,447 records containing at least one usable published price.

In [ ]:
# ============================================================
# STEP: Prepare all 3 hospitals in the same format
# ============================================================

import pandas as pd

# ---------- HOUSTON METHODIST ----------
methodist_rows = []

for record in methodist_data["standard_charge_information"]:
    service = record.get("description")

    for charge in record.get("standard_charges", []):
        payers = charge.get("payers_information", [])

        if not payers:
            methodist_rows.append({
                "hospital": "Houston Methodist",
                "service": service,
                "gross_price": charge.get("gross_charge"),
                "cash_price": charge.get("discounted_cash"),
                "payer": None,
                "plan": None,
                "negotiated_price": None
            })

        for payer in payers:
            methodist_rows.append({
                "hospital": "Houston Methodist",
                "service": service,
                "gross_price": charge.get("gross_charge"),
                "cash_price": charge.get("discounted_cash"),
                "payer": payer.get("payer_name"),
                "plan": payer.get("plan_name"),
                "negotiated_price": payer.get("standard_charge_dollar")
            })

methodist_clean = pd.DataFrame(methodist_rows)


# ---------- MEMORIAL HERMANN ----------
memorial_clean = pd.DataFrame({
    "hospital": "Memorial Hermann-Texas Medical Center",
    "service": memorial_df["description"],
    "gross_price": memorial_df["standard_charge|gross"],
    "cash_price": memorial_df["standard_charge|discounted_cash"],
    "payer": memorial_df["payer_name"],
    "plan": memorial_df["plan_name"],
    "negotiated_price": memorial_df["standard_charge|negotiated_dollar"]
})


# ---------- BAYLOR ST. LUKE'S ----------
baylor_rows = []

for record in baylor_data["standard_charge_information"]:
    service = record.get("description")

    for charge in record.get("standard_charges", []):
        payers = charge.get("payers_information", [])

        if not payers:
            baylor_rows.append({
                "hospital": "Baylor St. Luke's Medical Center",
                "service": service,
                "gross_price": charge.get("gross_charge"),
                "cash_price": charge.get("discounted_cash"),
                "payer": None,
                "plan": None,
                "negotiated_price": None
            })

        for payer in payers:
            baylor_rows.append({
                "hospital": "Baylor St. Luke's Medical Center",
                "service": service,
                "gross_price": charge.get("gross_charge"),
                "cash_price": charge.get("discounted_cash"),
                "payer": payer.get("payer_name"),
                "plan": payer.get("plan_name"),
                "negotiated_price": payer.get("standard_charge_dollar")
            })

baylor_clean = pd.DataFrame(baylor_rows)

print("All 3 hospital datasets prepared successfully!")
print("Methodist rows:", len(methodist_clean))
print("Memorial Hermann rows:", len(memorial_clean))
print("Baylor St. Luke's rows:", len(baylor_clean))
# Preserve billing codes for hospital comparisons

def extract_codes(record):
    codes = record.get("code_information", [])
    if isinstance(codes, list):
        return "; ".join(
            str(c.get("code", ""))
            for c in codes
            if isinstance(c, dict) and c.get("code")
        )
    return ""

# Match original records to the prepared rows
methodist_codes = {
    r.get("description"): extract_codes(r)
    for r in methodist_data["standard_charge_information"]
}

baylor_codes = {
    r.get("description"): extract_codes(r)
    for r in baylor_data["standard_charge_information"]
}

methodist_clean["billing_code"] = (
    methodist_clean["service"].map(methodist_codes)
)

baylor_clean["billing_code"] = (
    baylor_clean["service"].map(baylor_codes)
)

# Memorial Hermann billing codes
memorial_clean["billing_code"] = (
    memorial_df["code|1"].values
)

for name, df in [
    ("Methodist", methodist_clean),
    ("Memorial", memorial_clean),
    ("Baylor", baylor_clean)
]:
    print(name, "billing codes available:",
          df["billing_code"].notna().sum())


All 3 hospital datasets prepared successfully!
Methodist rows: 274797
Memorial Hermann rows: 131481
Baylor St. Luke's rows: 1823156
Methodist billing codes available: 274797
Memorial billing codes available: 130841
Baylor billing codes available: 1823156


In [ ]:
# Combine all 3 hospitals into one CareCost dataset

carecost_df = pd.concat(
    [methodist_clean, memorial_clean, baylor_clean],
    ignore_index=True
)

print("CareCost combined dataset created successfully!")
print("Total rows:", len(carecost_df))

print("\nRows by hospital:")
print(carecost_df["hospital"].value_counts())

print("\nColumns:")
print(carecost_df.columns.tolist())


# Verify billing codes in the combined dataset

print("\nBilling codes available:",
      carecost_df["billing_code"].notna().sum())

display(
    carecost_df[
        ["hospital", "service", "billing_code"]
    ].head(10)
)
# Check CT Chest services across all three hospitals

ct_check = carecost_df[
    carecost_df["service"].str.contains(
        "CT CHEST", case=False, na=False
    )
]

display(
    ct_check[
        ["hospital", "service", "billing_code", "cash_price"]
    ].drop_duplicates().groupby("hospital").head(5)
)


CareCost combined dataset created successfully!
Total rows: 2229434

Rows by hospital:
hospital
Baylor St. Luke's Medical Center         1823156
Houston Methodist                         274797
Memorial Hermann-Texas Medical Center     131481
Name: count, dtype: int64

Columns:
['hospital', 'service', 'gross_price', 'cash_price', 'payer', 'plan', 'negotiated_price', 'billing_code']

Billing codes available: 2228794


,hospital,service,billing_code
0,Houston Methodist,AMOXICILLIN 250 MG-POTASSIUM CLAVULANATE 125 M...,42571016030
1,Houston Methodist,FOLIC ACID 400 MCG TABLET [3234],50268034611
2,Houston Methodist,FOLIC ACID 5 MG/ML INJECTION SOLUTION [3232],63323018410
3,Houston Methodist,FOLIC ACID 5 MG/ML INJECTION SOLUTION [3232],63323018410
4,Houston Methodist,FOLIC ACID 5 MG/ML INJECTION SOLUTION [3232],63323018410
5,Houston Methodist,FOMEPIZOLE 1 GRAM/ML INTRAVENOUS SOLUTION [22185],67457021102
6,Houston Methodist,FOMEPIZOLE 1 GRAM/ML INTRAVENOUS SOLUTION [22185],67457021102
7,Houston Methodist,FOMEPIZOLE 1 GRAM/ML INTRAVENOUS SOLUTION [22185],67457021102
8,Houston Methodist,FONDAPARINUX 10 MG/0.8 ML SUBCUTANEOUS SOLUTIO...,67457058508
9,Houston Methodist,FONDAPARINUX 10 MG/0.8 ML SUBCUTANEOUS SOLUTIO...,67457058508


,hospital,service,billing_code,cash_price
12733,Houston Methodist,HC CT CHEST W CONTRAST,35200005; 0352; 71260,1197.50
12734,Houston Methodist,HC CT CHEST W/WO CONTRAST,35200006; 0352; 71270,1317.00
12766,Houston Methodist,HC CT CHEST WO CONTRAST,35200039; 0352; 71250,1088.50
28242,Houston Methodist,Ct chest spine w/o dye,72128,NaN
28256,Houston Methodist,Ct chest spine w/dye,72129,NaN
283311,Memorial Hermann-Texas Medical Center,"HC CT SCAN,THORAX,W/O CONTRAST - CT CHEST WO C...",3527125001,1346.80
283312,Memorial Hermann-Texas Medical Center,HC CAT SCAN OF CHEST CONTRAST - CT CHEST W CON...,3527126002,1659.12
283313,Memorial Hermann-Texas Medical Center,HC CAT SCAN OF CHEST COMBO - CT CHEST W WO CON...,3527127001,2165.28
283314,Memorial Hermann-Texas Medical Center,"HC CT ANGIO, CHEST, COMBO, INCL IMAGE - CT CHE...",3527127501,1601.92
320134,Memorial Hermann-Texas Medical Center,Ct chest spine w/o dye,72128,NaN


In [ ]:
# Check how our target services appear in the hospital datasets

keywords = ["MRI", "CT", "X-RAY", "COLONOSCOPY", "EMERGENCY"]

for keyword in keywords:
    matches = carecost_df[
        carecost_df["service"].astype(str).str.contains(
            keyword, case=False, na=False
        )
    ]

    print(f"\n--- {keyword} ---")
    print("Matching rows:", len(matches))

    # Show a few unique service names
    for service in matches["service"].dropna().unique()[:5]:
        print("•", service)


--- MRI ---
Matching rows: 8059
• HC MARKER BIOPSY BREAST MRI 14GA TITANIUM
• HC TISSUE MARKER BREAST MRI M-SHAPED SS 14GA
• HC MARKER TISS BRST MRI MSHPD SS 14GA SENOMARK
• HC INJ FOR KNEE ARTHROGRAM CT/MRI
• HC FUSING MRI PET MED DOSIMETRIST

--- CT ---
Matching rows: 217717
• FOLIC ACID 5 MG/ML INJECTION SOLUTION [3232]
• FLUTICASONE PROPIONATE 110 MCG/ACTUATION HFA AEROSOL INHALER [40698]
• FLUTICASONE PROPIONATE 220 MCG/ACTUATION HFA AEROSOL INHALER [40699]
• FLUTICASONE PROPIONATE 44 MCG/ACTUATION HFA AEROSOL INHALER [40697]
• FLUTICASONE PROPIONATE 50 MCG/ACTUATION NASAL SPRAY,SUSPENSION [70536]

--- X-RAY ---
Matching rows: 5087
• HC X-RAY STRENOCLAVIC JT MIN 3VWS
• HC X-RAY EXAM ENTIRE SPINE 1 VIEW
• HC X-RAY EXAM ENTIRE SPINE 2 VIEW
• HC X-RAY EXAM HIP W PELVIS MIN 4 VWS
• HC X-RAY EXAM ESOPHAGUS SGL CONTRAST

--- COLONOSCOPY ---
Matching rows: 3328
• COLONOSCOPY PREP POWDER PK (SALINE/SOD BICARB/POT CHL) [525025]
• HC COLONOSCOPY STOMA W/BALLOON DILATION
• HC COLONOSCOPY ST

In [ ]:
# Refine searches for the 5 CareCost healthcare services

patterns = {
    "MRI": r"\bMRI\b",
    "CT Scan": r"\bCT\b.*(?:SCAN|HEAD|CHEST|ABD|PELVIS|SPINE)|(?:SCAN|HEAD|CHEST|ABD|PELVIS|SPINE).*\bCT\b",
    "X-Ray": r"\bX[\s-]?RAY\b",
    "Colonoscopy": r"\bCOLONOSCOPY\b(?!.*PREP)",
    "Emergency Department": r"\bEMERGENCY (?:ROOM|DEPARTMENT)\b"
}

for category, pattern in patterns.items():

    matches = carecost_df[
        carecost_df["service"].astype(str).str.contains(
            pattern, case=False, na=False, regex=True
        )
    ]

    print(f"\n--- {category} ---")
    print("Matching rows:", len(matches))

    for service in matches["service"].dropna().unique()[:8]:
        print("•", service)


--- MRI ---
Matching rows: 7376
• HC MARKER BIOPSY BREAST MRI 14GA TITANIUM
• HC TISSUE MARKER BREAST MRI M-SHAPED SS 14GA
• HC MARKER TISS BRST MRI MSHPD SS 14GA SENOMARK
• HC INJ FOR KNEE ARTHROGRAM CT/MRI
• HC FUSING MRI PET MED DOSIMETRIST
• HC PRE-PLAN 3D MODEL WITH CCTA AND/OR MRI
• HC BX PROSTATE EA ADDL MRI-US FUS/IN-BORE CT/MRI
• HC BX PROSTATE TRCT US W/MRI FUS 1ST LESION

--- CT Scan ---
Matching rows: 2698
• HC CT SCAN FOR NEEDLE BIOPSY
• HC AUGMNT ALYS CHEST CT ILD W/O CONCURRENT CT EXAM
• HC AUGMNT ALYS CHEST CT ILD W/CONCURRENT CT EXAM
• HC CT HEAD/BRAIN SCAN WO CONTRA
• HC CT HEAD SCAN WITH INFUSION
• HC CT HEAD/BRAIN SCAN WO/W CONTRAST
• HC CT CHEST W CONTRAST
• HC CT CHEST W/WO CONTRAST

--- X-Ray ---
Matching rows: 4870
• HC X-RAY STRENOCLAVIC JT MIN 3VWS
• HC X-RAY EXAM ENTIRE SPINE 1 VIEW
• HC X-RAY EXAM ENTIRE SPINE 2 VIEW
• HC X-RAY EXAM HIP W PELVIS MIN 4 VWS
• HC X-RAY EXAM ESOPHAGUS SGL CONTRAST
• HC X-RAY EXAM SWALLOW FUNCTION CONTRAST STUDY
• HC X-RAY EXAM 

In [ ]:
# Cell 23 - Select specific healthcare services

import pandas as pd

# Select representative services using precise search patterns
procedures = {
    "MRI Brain": r"\bMRI\b.*\b(BRAIN|HEAD)\b",
    "CT Chest": r"\bCT\b.*\bCHEST\b",
    "Chest X-Ray": r"\b(X-RAY|XRAY)\b.*\bCHEST\b",
    "Colonoscopy": r"\bCOLONOSCOPY WITH BIOPSY\b|\bCOLONOSCOPY W.*BIOPSY\b",
    "Emergency Visit": r"\bEMERGENCY DEPARTMENT LEVEL [1-5]\b"
}

selected_data = []

for category, pattern in procedures.items():

    matches = carecost_df[
        carecost_df["service"].astype(str).str.contains(
            pattern, case=False, na=False, regex=True
        )
    ].copy()

    matches["category"] = category
    selected_data.append(matches)

# Combine selected records
carecost_small = pd.concat(selected_data, ignore_index=True)

# Show how many records we found at each hospital
summary = carecost_small.groupby(
    ["category", "hospital"]
).size().unstack(fill_value=0)

print("Smaller dataset created!")
print("Total matching rows:", len(carecost_small))

display(summary)

/tmp/ipykernel_4322/3788145170.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  carecost_df["service"].astype(str).str.contains(
/tmp/ipykernel_4322/3788145170.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  carecost_df["service"].astype(str).str.contains(


Smaller dataset created!
Total matching rows: 759


hospital,Baylor St. Luke's Medical Center,Houston Methodist,Memorial Hermann-Texas Medical Center
category,,,
CT Chest,285,59,32
Chest X-Ray,0,90,44
Colonoscopy,84,21,8
Emergency Visit,0,0,5
MRI Brain,0,84,47


In [ ]:
# Cell 24 - Inspect procedure names across hospitals

for category in carecost_small["category"].unique():

    print(f"\n===== {category} =====")

    subset = carecost_small[
        carecost_small["category"] == category
    ]

    for hospital in subset["hospital"].unique():

        print(f"\n{hospital}")

        names = subset[
            subset["hospital"] == hospital
        ]["service"].dropna().unique()

        for name in names[:5]:
            print("•", name)


===== MRI Brain =====

Houston Methodist
• HC MRI BRAIN OPEN INTRCRL WO CONT
• HC MRI BRAIN W/O CONTRAST
• HC MRI BRAIN W CONTRAST
• HC MRI BRAIN/HEAD W/WO CONTRAST
• HC MRI BRAIN FUNC W/PHYS ADMN

Memorial Hermann-Texas Medical Center
• HC MRI BRAIN - MRI BRAIN WO CONTRAST
• HC MRI BRAIN CONTRAST - MRI BRAIN W CONTRAST
• HC MRI BRAIN COMBO - MRI BRAIN W WO CONTRAST
• HC FUNC MRI BRAIN - MR BRAIN NEUROFUNCTIONAL TEST W/O PHYSICIAN ADMIN
• HC FUNC MRI BRAIN - MR BRAIN NEUROFUNCTIONAL TEST W/ PHYSICIAN ADMIN

===== CT Chest =====

Houston Methodist
• HC CT CHEST W CONTRAST
• HC CT CHEST W/WO CONTRAST
• HC CT CHEST WO CONTRAST
• Ct angiography chest
• Ct chest spine w/o dye

Memorial Hermann-Texas Medical Center
• HC CT SCAN,THORAX,W/O CONTRAST - CT CHEST WO CONTRAST
• HC CAT SCAN OF CHEST CONTRAST - CT CHEST W CONTRAST
• HC CAT SCAN OF CHEST COMBO - CT CHEST W WO CONTRAST
• HC CT ANGIO, CHEST, COMBO, INCL IMAGE - CT CHEST ANGIO W AND WO IV CONT
• Ct angiography chest

Baylor St. Luke's 

In [ ]:
# Cell 25 - Preview hospital prices for selected services

price_preview = carecost_small[
    ["category", "hospital", "service",
     "cash_price", "payer", "plan", "negotiated_price"]
].copy()

# Convert prices to numbers
for column in ["cash_price", "negotiated_price"]:
    price_preview[column] = pd.to_numeric(
        price_preview[column], errors="coerce"
    )

# Show available price information
display(
    price_preview[
        price_preview["cash_price"].notna() |
        price_preview["negotiated_price"].notna()
    ].head(20)
)

,category,hospital,service,cash_price,payer,plan,negotiated_price
0,MRI Brain,Houston Methodist,HC MRI BRAIN OPEN INTRCRL WO CONT,2060.0,None,None,NaN
1,MRI Brain,Houston Methodist,HC MRI BRAIN W/O CONTRAST,1335.5,None,None,NaN
2,MRI Brain,Houston Methodist,HC MRI BRAIN W CONTRAST,1469.5,None,None,NaN
3,MRI Brain,Houston Methodist,HC MRI BRAIN/HEAD W/WO CONTRAST,1615.5,None,None,NaN
4,MRI Brain,Houston Methodist,HC MRI BRAIN FUNC W/PHYS ADMN,437.5,None,None,NaN
5,MRI Brain,Houston Methodist,HC MRI BRAIN OPEN INTRCRL W/WO CO,530.0,None,None,NaN
6,MRI Brain,Houston Methodist,Mri brain stem w/o dye,NaN,Aetna,Hmo/Pos/Ppo,543.93
7,MRI Brain,Houston Methodist,Mri brain stem w/o dye,NaN,Aetna,Medicare Managed Care - Hmo/Ppo,243.28
8,MRI Brain,Houston Methodist,Mri brain stem w/o dye,NaN,Aetna,New Business Discount Other Commercial Plan,527.45
9,MRI Brain,Houston Methodist,Mri brain stem w/o dye,NaN,Bcbs,All Commercial Plans,1691.87


In [ ]:
# Cell 26 - Compare CT Chest prices across hospitals

ct_prices = price_preview[
    (price_preview["category"] == "CT Chest") &
    (price_preview["service"].str.contains(
        r"\bCT\b.*\bCHEST\b.*(?:W/O|WO|WITHOUT)\s+CONTRAST",
        case=False, na=False, regex=True
    ))
].copy()

print("Matching CT Chest without contrast records:",
      len(ct_prices))

display(
    ct_prices[
        ["hospital", "service", "cash_price",
         "payer", "plan", "negotiated_price"]
    ].head(30)
)

print("\nRecords by hospital:")
print(ct_prices["hospital"].value_counts())

Matching CT Chest without contrast records: 99


,hospital,service,cash_price,payer,plan,negotiated_price
132,Houston Methodist,HC CT CHEST W/WO CONTRAST,1317.00,None,None,NaN
133,Houston Methodist,HC CT CHEST WO CONTRAST,1088.50,None,None,NaN
190,Memorial Hermann-Texas Medical Center,"HC CT SCAN,THORAX,W/O CONTRAST - CT CHEST WO C...",1346.80,NaN,NaN,NaN
192,Memorial Hermann-Texas Medical Center,HC CAT SCAN OF CHEST COMBO - CT CHEST W WO CON...,2165.28,NaN,NaN,NaN
222,Baylor St. Luke's Medical Center,HC CT CHEST WO CONTRAST,2796.85,Cigna,Commercial|Transplant,5593.70
223,Baylor St. Luke's Medical Center,HC CT CHEST WO CONTRAST,2796.85,Coventry,Commercial|PPO,5194.15
224,Baylor St. Luke's Medical Center,HC CT CHEST WO CONTRAST,2796.85,First Health,Commercial|All Plans,5194.15
225,Baylor St. Luke's Medical Center,HC CT CHEST WO CONTRAST,2796.85,Healthsmart,Commercial|PPO,5593.70
226,Baylor St. Luke's Medical Center,HC CT CHEST WO CONTRAST,2796.85,Humana,Commercial|Transplant,5194.15
227,Baylor St. Luke's Medical Center,HC CT CHEST WO CONTRAST,2796.85,Interlink,Commercial|Transplant,5593.70



Records by hospital:
hospital
Baylor St. Luke's Medical Center         95
Houston Methodist                         2
Memorial Hermann-Texas Medical Center     2
Name: count, dtype: int64


In [ ]:
# CareCost AI - Final 5-treatment cash comparison

import pandas as pd

# Verified procedure records from our hospital data inspection
# Each entry: treatment, hospital, original dataset row index

verified_records = {
    "MRI Brain without contrast": {
        "Houston Methodist": 14376,
        "Memorial Hermann-Texas Medical Center": 284187,
        "Baylor St. Luke's Medical Center": 886807
    },
    "CT Chest without contrast": {
        "Houston Methodist": 12766,
        "Memorial Hermann-Texas Medical Center": 283311,
        "Baylor St. Luke's Medical Center": 888710
    },
    "Chest X-ray (2 views)": {
        "Houston Methodist": 12586,
        "Memorial Hermann-Texas Medical Center": 283148,
        "Baylor St. Luke's Medical Center": 887686
    },
    "Colonoscopy with biopsy": {
        "Houston Methodist": 13175,
        "Memorial Hermann-Texas Medical Center": 284414,
        "Baylor St. Luke's Medical Center": None
    },
    "Emergency Visit (Level 3)": {
        "Houston Methodist": 14021,
        "Memorial Hermann-Texas Medical Center": 283899,
        "Baylor St. Luke's Medical Center": None
    }
}

rows = []
audit_rows = []

for treatment, hospital_records in verified_records.items():
    row = {"Treatment": treatment}

    for hospital, index in hospital_records.items():
        price = None

        if index is not None:
            record = carecost_df.loc[index]

            if record["hospital"] != hospital:
                raise ValueError(
                    f"Hospital mismatch for {treatment}: {hospital}"
                )

            price = pd.to_numeric(
                record["cash_price"], errors="coerce"
            )

            audit_rows.append({
                "Treatment": treatment,
                "Hospital": hospital,
                "Service": record["service"],
                "Billing Code": record["billing_code"],
                "Cash Price": price
            })

        row[hospital] = price

    rows.append(row)

cash_comparison = pd.DataFrame(rows).set_index("Treatment")

# Display the final comparison horizontally
# Shorten hospital headings
clean_table = cash_comparison.copy()

clean_table.columns = [
    "Methodist",
    "Memorial Hermann",
    "Baylor St. Luke's"
]

# Display a compact, aligned table
display(
    clean_table.style
    .format("${:,.2f}", na_rep="N/A")
    .set_table_styles([
        {"selector": "th", "props": [
            ("text-align", "center"),
            ("padding", "8px 12px")
        ]},
        {"selector": "td", "props": [
            ("text-align", "center"),
            ("padding", "8px 12px")
        ]},
        {"selector": "table", "props": [
            ("width", "auto"),
            ("table-layout", "auto")
        ]}
    ])
)
# Keep audit records for verification
cash_price_audit = pd.DataFrame(audit_rows)

,Methodist,Memorial Hermann,Baylor St. Luke's
Treatment,,,
MRI Brain without contrast,"$1,335.50","$1,667.36","$1,647.80"
CT Chest without contrast,"$1,088.50","$1,346.80","$2,796.85"
Chest X-ray (2 views),$625.00,$389.28,$572.25
Colonoscopy with biopsy,$462.00,"$1,541.44",N/A
Emergency Visit (Level 3),"$1,121.00",$823.60,N/A


In [ ]:
# Cell 29 - Prepare cash and insurance prices

import pandas as pd

carecost_prices = carecost_df.copy()

# Convert prices to numbers
for col in ["cash_price", "negotiated_price"]:
    carecost_prices[col] = pd.to_numeric(
        carecost_prices[col], errors="coerce"
    )

# Keep records with at least one published price
carecost_prices = carecost_prices[
    carecost_prices["cash_price"].notna() |
    carecost_prices["negotiated_price"].notna()
].copy()

# Preserve missing insurance information
carecost_prices["payer"] = carecost_prices["payer"].fillna("Not specified")
carecost_prices["plan"] = carecost_prices["plan"].fillna("Not specified")

print("Usable pricing records:", len(carecost_prices))
print("Cash prices available:", carecost_prices["cash_price"].notna().sum())
print("Insurance rates available:", carecost_prices["negotiated_price"].notna().sum())

Usable pricing records: 1997447
Cash prices available: 1440308
Insurance rates available: 1969686


### Step 4: Hospital Cash and Insurance Price Comparison

**Objective:** Compare five medical services across three Houston hospitals using published price-transparency data.

**Method:** Python, Pandas, and interactive dropdowns (`ipywidgets`) filter prices by treatment (CPT code), insurance company, and plan.

**Output:** A side-by-side comparison of cash prices or available insurance-negotiated rates.

**Note:** N/A means no matching rate was found. Insurance rates are not the patient's final bill; out-of-pocket costs are estimated in the next step.

In [ ]:
# Cell 30 - Interactive cash and insurance comparison

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

services = {
    "MRI Brain without contrast": "70551",
    "CT Chest without contrast": "71250",
    "Chest X-ray (2 views)": "71046",
    "Colonoscopy with biopsy": "45380",
    "Emergency Visit (Level 3)": "99283"
}

hospitals = [
    "Houston Methodist",
    "Memorial Hermann-Texas Medical Center",
    "Baylor St. Luke's Medical Center"
]

short_names = {
    hospitals[0]: "Methodist",
    hospitals[1]: "Memorial Hermann",
    hospitals[2]: "Baylor St. Luke's"
}

# Find insurance rates using CPT codes
insurance_rows = []

for treatment, cpt in services.items():
    matches = carecost_prices[
        carecost_prices["hospital"].isin(hospitals) &
        carecost_prices["billing_code"].fillna("")
            .astype(str).str.contains(cpt, regex=False) &
        carecost_prices["negotiated_price"].notna()
    ].copy()

    matches["Treatment"] = treatment
    insurance_rows.append(matches)

insurance_df = pd.concat(insurance_rows, ignore_index=True)

# Standardize insurance company names
insurance_df["payer"] = (
    insurance_df["payer"].fillna("Not specified")
    .astype(str).str.strip().str.upper()
)

insurance_df["plan"] = (
    insurance_df["plan"].fillna("Not specified")
    .astype(str).str.strip()
)

# Interactive dropdowns
service_choice = widgets.Dropdown(
    options=list(services.keys()),
    description="Treatment:",
    style={"description_width": "initial"}
)

payer_choice = widgets.Dropdown(
    options=["Self-pay"],
    description="Insurance:",
    style={"description_width": "initial"}
)

plan_choice = widgets.Dropdown(
    options=["Not applicable"],
    description="Plan:",
    style={"description_width": "initial"}
)

output = widgets.Output()

def update_payers(change=None):
    treatment = service_choice.value

    payers = sorted(
        insurance_df.loc[
            insurance_df["Treatment"] == treatment,
            "payer"
        ].dropna().unique().tolist()
    )

    payer_choice.options = ["Self-pay"] + payers
    update_plans()


def update_plans(change=None):
    if payer_choice.value == "Self-pay":
        plan_choice.options = ["Not applicable"]
    else:
        plans = sorted(
            insurance_df.loc[
                (insurance_df["Treatment"] == service_choice.value) &
                (insurance_df["payer"] == payer_choice.value),
                "plan"
            ].dropna().unique().tolist()
        )

        plan_choice.options = plans if plans else ["No matching plan"]

    show_comparison()


def show_comparison(change=None):
    with output:
        clear_output(wait=True)

        treatment = service_choice.value

        if payer_choice.value == "Self-pay":
            prices = cash_comparison.loc[[treatment]].copy()
            prices = prices.reindex(columns=hospitals)

        else:
            matches = insurance_df[
                (insurance_df["Treatment"] == treatment) &
                (insurance_df["payer"] == payer_choice.value) &
                (insurance_df["plan"] == plan_choice.value)
            ]

            if matches.empty:
                prices = pd.DataFrame(
                    index=[treatment],
                    columns=hospitals,
                    dtype=float
                )
            else:
                prices = matches.pivot_table(
                    index="Treatment",
                    columns="hospital",
                    values="negotiated_price",
                    aggfunc="min"
                ).reindex(
                    index=[treatment],
                    columns=hospitals
                )

        prices = prices.rename(columns=short_names)

        print("HOSPITAL PRICE COMPARISON")
        display(prices.style.format("${:,.2f}", na_rep="N/A"))

        # Show available plans for the selected insurer
        if payer_choice.value != "Self-pay":
            available = insurance_df[
                (insurance_df["Treatment"] == treatment) &
                (insurance_df["payer"] == payer_choice.value)
            ]

            plan_summary = (
                available.groupby(
                    ["hospital", "plan"], as_index=False
                )["negotiated_price"].min()
            )

            plan_summary["hospital"] = (
                plan_summary["hospital"].replace(short_names)
            )

            plan_summary = plan_summary.rename(columns={
                "hospital": "Hospital",
                "plan": "Insurance Plan",
                "negotiated_price": "Published Rate"
            })

            print("\nAVAILABLE INSURANCE PLANS AND RATES")

            if plan_summary.empty:
                print("No matching rates found.")
            else:
                display(
                    plan_summary.style.format(
                        {"Published Rate": "${:,.2f}"}
                    )
                )

        print("\nN/A = No matching published rate found.")
        print("Different plans may have different rates.")
        print("Insurance rates are not final patient bills.")
        print("Procedure matches require verification.")


# Connect dropdowns
service_choice.observe(update_payers, names="value")
payer_choice.observe(update_plans, names="value")
plan_choice.observe(show_comparison, names="value")

# Display controls OUTSIDE the functions
display(service_choice, payer_choice, plan_choice, output)
update_payers()

HOSPITAL PRICE COMPARISON


hospital,Methodist,Memorial Hermann,Baylor St. Luke's
Treatment,,,
CT Chest without contrast,N/A,N/A,$164.03



AVAILABLE INSURANCE PLANS AND RATES


,Hospital,Insurance Plan,Published Rate
0,Baylor St. Luke's,Commercial|Exchange,$164.03
1,Baylor St. Luke's,Commercial|HMO,"$5,593.70"
2,Baylor St. Luke's,Commercial|KelseyCare,"$4,874.51"
3,Baylor St. Luke's,Commercial|PPO,$761.00
4,Baylor St. Luke's,Commercial|Transplant,$590.00
5,Baylor St. Luke's,Medicare|All Other Plans,$108.64
6,Baylor St. Luke's,Medicare|TRANSPLANT,$106.51
7,Methodist,Hmo/Pos/Ppo,$352.68
8,Methodist,Medicare Managed Care - Hmo/Ppo,$106.60
9,Methodist,New Business Discount Other Commercial Plan,$341.99



N/A = No matching published rate found.
Different plans may have different rates.
Insurance rates are not final patient bills.
Procedure matches require verification.


In [ ]:
# Check original MRI insurance records

for hospital in hospitals:
    records = carecost_prices[
        (carecost_prices["hospital"] == hospital) &
        (carecost_prices["billing_code"].fillna("")
         .astype(str).str.contains("70551", regex=False)) &
        (carecost_prices["negotiated_price"].notna())
    ]

    print("\nHospital:", hospital)
    print("MRI CPT 70551 insurance records:", len(records))
    print(
        records[["service", "payer", "plan", "negotiated_price"]]
        .head(5).to_string(index=False)
    )


Hospital: Houston Methodist
MRI CPT 70551 insurance records: 14
               service payer                                        plan  negotiated_price
Mri brain stem w/o dye Aetna                                 Hmo/Pos/Ppo            543.93
Mri brain stem w/o dye Aetna             Medicare Managed Care - Hmo/Ppo            243.28
Mri brain stem w/o dye Aetna New Business Discount Other Commercial Plan            527.45
Mri brain stem w/o dye  Bcbs                        All Commercial Plans           1691.87
Mri brain stem w/o dye  Bcbs                 Medicare Managed Care - Hmo            239.69

Hospital: Memorial Hermann-Texas Medical Center
MRI CPT 70551 insurance records: 7
               service            payer                       plan  negotiated_price
Mri brain stem w/o dye       90 Degrees       All Commercial Plans            455.41
Mri brain stem w/o dye            Aetna                   Exchange            371.52
Mri brain stem w/o dye            Aetna Medicare M

### Step 5: Estimated Patient Out-of-Pocket Cost

**Objective:** Estimate what a patient may pay after insurance.

**Method:** Use the selected hospital's published insurance rate and patient-entered deductible, copay, and coinsurance.

**Output:** Estimated patient payment for the selected treatment and insurance plan.

**Limitation:** This is an educational estimate, not a guaranteed medical bill. Coverage and insurance benefits must be verified.

In [ ]:
# Cell 31 - Complete Patient Out-of-Pocket Calculator

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Use the insurance data already prepared in Cell 30
patient_data = insurance_df.copy()

# Make sure prices are numeric
patient_data["negotiated_price"] = pd.to_numeric(
    patient_data["negotiated_price"], errors="coerce"
)

patient_data = patient_data[
    patient_data["negotiated_price"].notna() &
    (patient_data["negotiated_price"] >= 0)
].copy()

# Patient selection fields
treatment_input = widgets.Dropdown(
    options=list(services.keys()),
    description="Treatment:",
    style={"description_width": "initial"}
)

hospital_input = widgets.Dropdown(
    options=[(short_names[h], h) for h in hospitals],
    description="Hospital:",
    style={"description_width": "initial"}
)

insurance_input = widgets.Dropdown(
    description="Insurance:",
    style={"description_width": "initial"}
)

plan_input = widgets.Dropdown(
    description="Plan:",
    style={"description_width": "initial"}
)

# Patient enters benefits from their insurance plan
deductible_input = widgets.BoundedFloatText(
    value=0, min=0,
    description="Deductible left ($):",
    style={"description_width": "initial"}
)

copay_input = widgets.BoundedFloatText(
    value=0, min=0,
    description="Copay ($):",
    style={"description_width": "initial"}
)

coinsurance_input = widgets.BoundedFloatText(
    value=0, min=0, max=100,
    description="Coinsurance (%):",
    style={"description_width": "initial"}
)

calculate_button = widgets.Button(
    description="Calculate My Cost",
    button_style="success"
)

cost_output = widgets.Output()

# Update available insurance companies
def update_insurance(change=None):
    filtered = patient_data[
        (patient_data["Treatment"] == treatment_input.value) &
        (patient_data["hospital"] == hospital_input.value)
    ]

    options = sorted(filtered["payer"].dropna().unique().tolist())
    insurance_input.options = options
    update_patient_plans()

# Update plans for selected hospital and insurance
def update_patient_plans(change=None):
    filtered = patient_data[
        (patient_data["Treatment"] == treatment_input.value) &
        (patient_data["hospital"] == hospital_input.value) &
        (patient_data["payer"] == insurance_input.value)
    ]

    options = sorted(filtered["plan"].dropna().unique().tolist())
    plan_input.options = options

# Calculate estimated patient cost
def calculate_patient_cost(button):
    with cost_output:
        clear_output(wait=True)

        filtered = patient_data[
            (patient_data["Treatment"] == treatment_input.value) &
            (patient_data["hospital"] == hospital_input.value) &
            (patient_data["payer"] == insurance_input.value) &
            (patient_data["plan"] == plan_input.value)
        ]

        if filtered.empty:
            print("No matching published rate found.")
            return

        # Same minimum-rate rule as Cell 30
        rate = float(filtered["negotiated_price"].min())

        deductible = min(rate, deductible_input.value)
        remaining = rate - deductible

        coinsurance = (
            remaining * coinsurance_input.value / 100
        )

        patient_cost = min(
            rate,
            deductible + coinsurance + copay_input.value
        )

        print("CARE COST AI - PATIENT ESTIMATE")
        print("--------------------------------")
        print("Treatment:", treatment_input.value)
        print("Hospital:", short_names[hospital_input.value])
        print("Insurance:", insurance_input.value)
        print("Plan:", plan_input.value)
        print(f"Published negotiated rate: ${rate:,.2f}")
        print(f"Estimated patient payment: ${patient_cost:,.2f}")
        print("--------------------------------")
        print("Educational estimate only.")
        print("Assumes covered, in-network care.")
        print("Actual benefits, copay rules and costs may differ.")
        print("Verify coverage and pricing with your insurer.")

# Connect dropdowns
treatment_input.observe(update_insurance, names="value")
hospital_input.observe(update_insurance, names="value")
insurance_input.observe(update_patient_plans, names="value")

calculate_button.on_click(calculate_patient_cost)

# Display complete calculator
display(
    widgets.HTML("<h3>CareCost AI - Patient Cost Calculator</h3>"),
    treatment_input,
    hospital_input,
    insurance_input,
    plan_input,
    deductible_input,
    copay_input,
    coinsurance_input,
    calculate_button,
    cost_output
)

update_insurance()

HTML(value='<h3>CareCost AI - Patient Cost Calculator</h3>')

Dropdown(description='Treatment:', options=('MRI Brain without contrast', 'CT Chest without contrast', 'Chest …

Dropdown(description='Hospital:', options=(('Methodist', 'Houston Methodist'), ('Memorial Hermann', 'Memorial …

Dropdown(description='Insurance:', options=(), style=DescriptionStyle(description_width='initial'), value=None…

Dropdown(description='Plan:', options=(), style=DescriptionStyle(description_width='initial'), value=None)

BoundedFloatText(value=0.0, description='Deductible left ($):', style=DescriptionStyle(description_width='init…

BoundedFloatText(value=0.0, description='Copay ($):', style=DescriptionStyle(description_width='initial'))

BoundedFloatText(value=0.0, description='Coinsurance (%):', style=DescriptionStyle(description_width='initial'…

Button(button_style='success', description='Calculate My Cost', style=ButtonStyle())

Output()

### Step 6: AI-Based Medical Service Matching

**Objective:** Understand a patient's medical question and identify the relevant service.

**Method:** A pretrained Sentence Transformer (`all-MiniLM-L6-v2`) converts text into embeddings and uses cosine similarity to match the question with five medical services.

**Output:** The closest matching service and its similarity score.

**Limitation:** The model identifies the most similar service; it does not diagnose medical conditions or verify insurance coverage.

In [ ]:
# Cell 32 - NLP service matching

%pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer, util

nlp_model = SentenceTransformer("all-MiniLM-L6-v2")

service_descriptions = {
    "MRI Brain": "MRI scan of the brain or head",
    "CT Chest": "CT scan of the chest or lungs",
    "Chest X-Ray": "Chest X-ray imaging examination",
    "Colonoscopy": "Colonoscopy examination of the colon",
    "Emergency Visit": "Emergency department hospital visit"
}

labels = list(service_descriptions.keys())

service_embeddings = nlp_model.encode(
    list(service_descriptions.values()),
    convert_to_tensor=True
)

def understand_question(question):
    question_embedding = nlp_model.encode(
        question, convert_to_tensor=True
    )

    scores = util.cos_sim(
        question_embedding, service_embeddings
    )[0]

    best_index = int(scores.argmax())

    return labels[best_index], round(
        float(scores[best_index]), 3
    )

print(understand_question("I need a CT scan of my lungs"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

('CT Chest', 0.837)


### Step 7: CareCost AI — Interactive Patient Interface

**Objective:** Help patients find and compare hospital prices using natural-language questions.

**How it works:** The NLP model identifies the requested medical service using sentence embeddings and cosine similarity. The application then displays published cash prices or insurance-negotiated rates across three Houston hospitals.

**Understanding the results:**
- **N/A:** No matching published rate was found for the selected insurance plan in our dataset. It does not mean the hospital does not accept that insurance.
- **Other Published Insurance Plans:** Shows additional plan names and rates reported by the hospitals for the selected insurer. These plans are not necessarily equivalent.
- **Displayed Price:** A published hospital rate, not necessarily the patient's final payment.

**Limitation:** Patients must confirm the exact procedure, insurance plan, network coverage, and final costs with their hospital and insurer.

In [ ]:

# Cell 33 - CareCost AI: Final User Interface

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Connect NLP results with our hospital services
service_map = {
    "MRI Brain": "MRI Brain without contrast",
    "CT Chest": "CT Chest without contrast",
    "Chest X-Ray": "Chest X-ray (2 views)",
    "Colonoscopy": "Colonoscopy with biopsy",
    "Emergency Visit": "Emergency Visit (Level 3)"
}

# Hospital display names
hospital_labels = {
    "Houston Methodist": "Methodist",
    "Memorial Hermann-Texas Medical Center": "Memorial Hermann",
    "Baylor St. Luke's Medical Center": "Baylor St. Luke's"
}

# Format money without turning missing values into zero
def money(value):
    return "N/A" if pd.isna(value) else f"${float(value):,.2f}"

question_box = widgets.Text(
    value="I need a CT scan of my lungs",
    description="Question:",
    placeholder="Describe the medical service you need",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "100px"}
)

payment_box = widgets.Dropdown(
    options=["Self-pay", "Insurance"],
    description="Payment:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="360px")
)

payer_box = widgets.Dropdown(
    description="Insurance:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="360px")
)

plan_box = widgets.Dropdown(
    description="Plan:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="480px")
)

search_button = widgets.Button(
    description="Compare Hospitals",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="200px")
)

app_output = widgets.Output()

def update_app_payers(change=None):
    insured = payment_box.value == "Insurance"
    payer_box.layout.display = "" if insured else "none"
    plan_box.layout.display = "" if insured else "none"

    if insured:
        payer_box.options = sorted(
            insurance_df["payer"].dropna().unique().tolist()
        )
        update_app_plans()

def update_app_plans(change=None):
    available = insurance_df[
        insurance_df["payer"] == payer_box.value
    ]
    plan_box.options = sorted(
        available["plan"].dropna().unique().tolist()
    )

def run_carecost(button):
    with app_output:
        clear_output(wait=True)

        question = question_box.value.strip()
        if not question:
            print("Please enter a medical-service question.")
            return

        label, similarity = understand_question(question)
        treatment = service_map[label]

        display(HTML("<h3>1. AI SERVICE MATCH</h3>"))
        summary = pd.DataFrame([{
            "Patient question": question,
            "Identified service": treatment,
            "Cosine similarity": f"{similarity:.3f}"
        }])
        display(summary.style.hide(axis="index"))

        # Safety: do not equate a cardiac CT with a routine chest CT
        if any(word in question.lower()
               for word in ["heart", "cardiac", "coronary"]):
            print(
                "A cardiac procedure may require a different CPT code. "
                "Please confirm the exact medical service."
            )
            return

        if similarity < 0.55:
            print(
                "The AI match is uncertain. "
                "Please provide a more specific procedure description."
            )
            return

        display(HTML("<h3>2. HOSPITAL PRICE COMPARISON</h3>"))

        if payment_box.value == "Self-pay":

            if treatment not in cash_comparison.index:
                print("No verified cash-price comparison available.")
                return

            # Treatment is stored in the row index
            comparison = cash_comparison.loc[[treatment]].copy()
            comparison = comparison.rename(columns=hospital_labels)

            # Display hospital columns in a consistent order
            hospital_order = [
                "Methodist", "Memorial Hermann", "Baylor St. Luke's"
            ]
            comparison = comparison.reindex(columns=hospital_order)
            comparison.index.name = "Medical service"

            display(comparison.style.format(money))

            display(HTML("<h3>3. PAYMENT INFORMATION</h3>"))
            print("Payment method: Self-pay (cash price)")
            print("N/A = No verified matching cash price found.")

        else:
            selected = insurance_df[
                (insurance_df["Treatment"] == treatment) &
                (insurance_df["payer"] == payer_box.value) &
                (insurance_df["plan"] == plan_box.value)
            ]

            rates = (
                selected.groupby("hospital")["negotiated_price"]
                .min()
                .reindex(hospitals)
            )

            # Make hospitals columns, with one row per service
            comparison = pd.DataFrame(
                [rates.to_numpy()],
                index=[treatment],
                columns=[hospital_labels.get(h, h) for h in hospitals]
            )
            comparison.index.name = "Medical service"

            display(comparison.style.format(money))

            display(HTML("<h3>3. INSURANCE DETAILS</h3>"))
            details = pd.DataFrame([{
                "Insurance company": payer_box.value,
                "Selected plan": plan_box.value
            }])
            display(details.style.hide(axis="index"))

            display(HTML("<h3>4. OTHER PUBLISHED INSURANCE PLANS</h3>"))

            other_plans = insurance_df[
                (insurance_df["Treatment"] == treatment) &
                (insurance_df["payer"] == payer_box.value)
            ]

            if other_plans.empty:
                print("No additional published insurance rates found.")
            else:
                plan_table = (
                    other_plans.groupby(["hospital", "plan"])[
                        "negotiated_price"
                    ]
                    .min()
                    .reset_index()
                    .rename(columns={
                        "hospital": "Hospital",
                        "plan": "Insurance plan",
                        "negotiated_price": "Published rate"
                    })
                )

                plan_table["Hospital"] = (
                    plan_table["Hospital"]
                    .replace(hospital_labels)
                )

                plan_table = plan_table.sort_values(
                    ["Hospital", "Insurance plan"]
                )

                display(
                    plan_table.style
                    .hide(axis="index")
                    .format({"Published rate": money})
                )

            print(
                "\nUse the calculator in Cell 31 to estimate "
                "your out-of-pocket payment."
            )

        display(HTML("<h3>IMPORTANT INFORMATION</h3>"))
        print(
            "• Cosine similarity is not model accuracy.\n"
            "• N/A means no verified matching published rate.\n"
            "• Different insurance plans are not interchangeable.\n"
            "• Published rates are not final patient bills.\n"
            "• Confirm the CPT code, procedure, insurance coverage, "
            "network status, and final price with the hospital and insurer."
        )

payment_box.observe(update_app_payers, names="value")
payer_box.observe(update_app_plans, names="value")
search_button.on_click(run_carecost)

display(HTML(
    "<h2>CareCost AI</h2>"
    "<p>Houston Healthcare Cost & Insurance Navigator</p>"
))

display(
    question_box,
    payment_box,
    payer_box,
    plan_box,
    search_button,
    app_output
)

update_app_payers()


Patient question,Identified service,Cosine similarity
I need a CT scan of my lungs,CT Chest without contrast,0.837


,Methodist,Memorial Hermann,Baylor St. Luke's
Medical service,,,
CT Chest without contrast,N/A,N/A,$164.03


Insurance company,Selected plan
AETNA,Commercial|Exchange


Hospital,Insurance plan,Published rate
Baylor St. Luke's,Commercial|Exchange,$164.03
Baylor St. Luke's,Commercial|HMO,"$5,593.70"
Baylor St. Luke's,Commercial|KelseyCare,"$4,874.51"
Baylor St. Luke's,Commercial|PPO,$761.00
Baylor St. Luke's,Commercial|Transplant,$590.00
Baylor St. Luke's,Medicare|All Other Plans,$108.64
Baylor St. Luke's,Medicare|TRANSPLANT,$106.51
Memorial Hermann,Exchange,$162.78
Memorial Hermann,Medicare Managed Care Plan,$105.02
Methodist,Hmo/Pos/Ppo,$352.68



Use the calculator in Cell 31 to estimate your out-of-pocket payment.


• Cosine similarity is not model accuracy.
• N/A means no verified matching published rate.
• Different insurance plans are not interchangeable.
• Published rates are not final patient bills.
• Confirm the CPT code, procedure, insurance coverage, network status, and final price with the hospital and insurer.


### Step 8: Application Testing

We tested the NLP model using five example patient questions covering MRI, CT, X-ray, colonoscopy, and emergency visits.

**Results:** All five service-matching tests passed (5/5). Two simplified patient cost-calculation tests also passed.

These are basic functionality tests, not a comprehensive evaluation of the model's accuracy or medical reliability.

In [ ]:

# Cell 34 - Final NLP and Cost Calculator Tests

test_cases = {
    "I need an MRI of my brain": "MRI Brain",
    "Compare CT chest scans": "CT Chest",
    "How much is a chest X-ray?": "Chest X-Ray",
    "I need a colonoscopy": "Colonoscopy",
    "I need an emergency department visit": "Emergency Visit"
}

correct = 0

for question, expected in test_cases.items():
    predicted, similarity = understand_question(question)
    passed = predicted == expected
    correct += int(passed)

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{question} -> {predicted} "
        f"(similarity: {similarity:.3f})"
    )

print(f"\nNLP test accuracy: {correct}/{len(test_cases)}")

# Same simplified formula used in Cell 31
def test_patient_cost(rate, deductible_left, copay, coinsurance):
    deductible_paid = min(rate, deductible_left)
    remaining = rate - deductible_paid
    return min(rate, deductible_paid + remaining * coinsurance / 100 + copay)

assert test_patient_cost(2000, 500, 0, 20) == 800
assert test_patient_cost(1000, 0, 0, 20) == 200

print("Cost calculation tests passed.")


PASS | I need an MRI of my brain -> MRI Brain (similarity: 0.759)
PASS | Compare CT chest scans -> CT Chest (similarity: 0.815)
PASS | How much is a chest X-ray? -> Chest X-Ray (similarity: 0.704)
PASS | I need a colonoscopy -> Colonoscopy (similarity: 0.770)
PASS | I need an emergency department visit -> Emergency Visit (similarity: 0.832)

NLP test accuracy: 5/5
Cost calculation tests passed.


### Step 8: Project Summary and Conclusion

**Project:** CareCost AI — Houston Healthcare Cost & Insurance Navigator

**Purpose:** Help patients understand and compare healthcare costs before receiving medical services.

**Implementation:** We used Python, Pandas, Google Colab, and a pretrained Sentence Transformer (`all-MiniLM-L6-v2`) to build an NLP-powered healthcare cost comparison application.

**Results:** The prototype compares five medical services across three Houston hospitals, displays published cash and insurance-negotiated rates, and provides an illustrative patient out-of-pocket cost calculator.

**Limitations:** The application covers a limited number of hospitals and procedures. NLP matches, published prices, insurance coverage, and patient cost estimates require verification.

**Future Improvements:** Expand hospital coverage, improve medical-service matching, and connect to regularly updated price-transparency data.